# Aprendizado de Máquina — Aula prática 10

## Máquinas de Vetores de Suporte

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Todos os classificadores das Aulas 08 e 09 partiam de uma probabilidade: modelavam
$P(Y=1\mid x)$ e cortavam. A SVM parte de outro lugar, puramente geométrico:

> **entre todas as fronteiras que separam as classes, escolha a que fica mais
> longe dos pontos.**

Dessa ideia saem três coisas que valem o notebook inteiro: a fronteira depende
apenas de **alguns** pontos (os vetores de suporte); a fronteira pode virar curva
sem que ninguém calcule coordenada nova (o truque do *kernel*); e o método não
entrega probabilidade nenhuma — o que, depois da Aula 09, você já sabe que é uma
limitação séria e não um detalhe.

### Objetivos

Ao final deste notebook você deve ser capaz de:

- identificar os vetores de suporte e **verificar** que só eles determinam a
  fronteira;
- explicar o que o $C$ controla e o que acontece com o número de vetores de suporte;
- **conferir numericamente o truque do kernel**, comparando o kernel polinomial com
  a expansão explícita de atributos;
- buscar $C$ e $\gamma$ em grade bidimensional e enxergar por que ela precisa ser
  bidimensional;
- comparar as perdas *hinge*, logística e 0–1;
- medir o custo computacional do `SVC` e confirmar por que ele não escala.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

Os objetos novos são o `SVC` e o `LinearSVC`, mais três geradores de dados
artificiais do `sklearn.datasets` — é com eles que as fronteiras ficam visíveis.

In [ ]:
import sklearn.model_selection as skm
from sklearn.datasets import make_blobs, make_circles, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import SVC, LinearSVC

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def desenhar_fronteira(ax, modelo, X, y, titulo="", margens=False, suporte=False):
    passo = 0.02
    x0 = np.arange(X[:, 0].min() - 0.7, X[:, 0].max() + 0.7, passo)
    x1 = np.arange(X[:, 1].min() - 0.7, X[:, 1].max() + 0.7, passo)
    gx, gy = np.meshgrid(x0, x1)
    Z = modelo.decision_function(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    ax.contourf(gx, gy, Z > 0, levels=[-0.5, 0.5, 1.5],
                colors=["steelblue", "crimson"], alpha=0.10)
    niveis = [-1, 0, 1] if margens else [0]
    estilos = ["--", "-", "--"] if margens else ["-"]
    ax.contour(gx, gy, Z, levels=niveis, colors="black", linestyles=estilos,
               linewidths=[1, 1.6, 1] if margens else [1.6])
    ax.scatter(X[y == 0, 0], X[y == 0, 1], s=18, color="steelblue", edgecolor="k", lw=0.3)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], s=18, color="crimson", edgecolor="k", lw=0.3)
    if suporte:
        sv = modelo.support_vectors_ if hasattr(modelo, "support_vectors_") \
            else modelo[-1].support_vectors_
        ax.scatter(sv[:, 0], sv[:, 1], s=110, facecolors="none", edgecolors="green",
                   linewidths=1.3)
    ax.set_title(titulo, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

---
## 2. A margem, e os pontos que a seguram

Comecemos pelo caso fácil: duas classes que uma reta separa. Há infinitas retas que
separam; a SVM escolhe a que maximiza a **margem**, a faixa vazia em torno da
fronteira.

In [ ]:
X, y = make_blobs(n_samples=60, centers=2, cluster_std=1.05, random_state=6)

svm = SVC(kernel="linear", C=1000).fit(X, y)      # C enorme = margem rigida
fig, ax = subplots(figsize=(5.0, 4.0))
desenhar_fronteira(ax, svm, X, y, "margem maxima (C grande)", margens=True, suporte=True)

print(f"observacoes: {len(y)}")
print(f"vetores de suporte: {svm.n_support_.sum()}  (circulados em verde)")
print(f"indices: {svm.support_}")

Três pontos, de sessenta, seguram a fronteira. Isso não é uma descrição vaga — é
literal, e dá para provar removendo pontos.

In [ ]:
coef_original = np.r_[svm.coef_.ravel(), svm.intercept_]

# (a) remover TODOS os pontos que nao sao vetores de suporte
mantidos = np.zeros(len(y), dtype=bool)
mantidos[svm.support_] = True
svm_so_sv = SVC(kernel="linear", C=1000).fit(X[mantidos], y[mantidos])
d_a = np.abs(np.r_[svm_so_sv.coef_.ravel(), svm_so_sv.intercept_] - coef_original).max()

# (b) remover UM vetor de suporte
fora = np.ones(len(y), dtype=bool)
fora[svm.support_[0]] = False
svm_sem_um = SVC(kernel="linear", C=1000).fit(X[fora], y[fora])
d_b = np.abs(np.r_[svm_sem_um.coef_.ravel(), svm_sem_um.intercept_] - coef_original).max()

print(f"jogando fora as {int((~mantidos).sum())} observacoes que NAO sao vetores de suporte:")
print(f"   maior mudanca nos coeficientes: {d_a:.2e}")
print(f"jogando fora UM vetor de suporte:")
print(f"   maior mudanca nos coeficientes: {d_b:.4f}")

Descartar 57 das 60 observações mexe na fronteira na **terceira casa decimal** — e
o que sobra é tolerância do otimizador, não estrutura. Descartar um único dos três
vetores de suporte muda mais de cem vezes isso.

Isso separa a SVM de quase todo o resto do curso. A regressão logística usa cada
observação — mover um ponto longe da fronteira muda (pouco, mas muda) os
coeficientes. A SVM ignora completamente quem já está do lado certo e com folga.
É uma virtude, porque dá robustez a pontos distantes, e é um risco, porque toda a
decisão fica nas mãos de pouquíssimas observações — inclusive de eventuais erros de
rotulagem bem na fronteira.

---
## 3. O $C$, quando as classes se misturam

Dados reais não se separam. A margem passa a ser **flexível**: permite-se que
alguns pontos invadam a faixa, ou até fiquem do lado errado, pagando uma
penalidade. O $C$ é o preço dessa invasão.

- $C$ **grande**: invadir é caro $\Rightarrow$ margem estreita, poucos vetores de
  suporte, fronteira que se contorce para acertar todo mundo (variância alta);
- $C$ **pequeno**: invadir é barato $\Rightarrow$ margem larga, muitos vetores de
  suporte, fronteira mais rígida (viés alto).

In [ ]:
Xm, ym = make_blobs(n_samples=200, centers=2, cluster_std=2.4, random_state=3)

fig, axes = subplots(1, 4, figsize=(11, 3.0))
linhas = []
for ax, c in zip(axes, [0.01, 0.1, 1, 100]):
    m = SVC(kernel="linear", C=c).fit(Xm, ym)
    desenhar_fronteira(ax, m, Xm, ym, f"C = {c}", margens=True, suporte=True)
    linhas.append({"C": c, "vetores de suporte": int(m.n_support_.sum()),
                   "acuracia no treino": m.score(Xm, ym),
                   "largura da margem": 2 / np.linalg.norm(m.coef_)})
pd.DataFrame(linhas).set_index("C").round(4)

A tabela mostra os dois efeitos ao mesmo tempo: à medida que $C$ cresce, a margem
encolhe e o número de vetores de suporte cai. São a mesma coisa vista de dois
ângulos — vetores de suporte são exatamente os pontos que estão sobre a margem ou
dentro dela.

Repare também que, de $C=1$ em diante, nada mais muda. A partir daí a penalidade já
é alta o bastante para o modelo fazer tudo o que consegue fazer, e os pontos que
invadem a margem invadem porque **não há reta** que os coloque do lado certo.
Aumentar mais o $C$ não tem contra o que lutar.

Note o que isso implica para o custo: com $C$ pequeno o modelo guarda quase todas
as observações. "Guardar" aqui é literal — a predição da SVM é uma soma sobre os
vetores de suporte.

---
## 4. O truque do *kernel*, verificado

Uma fronteira reta não serve para tudo. A saída clássica seria criar atributos
novos — $x_1^2$, $x_1x_2$, $x_2^2$ — e traçar uma reta no espaço ampliado, que vira
uma curva no original. O problema é que o número de atributos explode.

O **truque do kernel** evita isso. A SVM só precisa de **produtos internos** entre
observações, e um kernel calcula o produto interno no espaço ampliado **sem nunca
construir as coordenadas**. Para o kernel polinomial de grau 2,

$$K(x, x') = (\gamma\,\langle x, x'\rangle + r)^2 = \langle \phi(x), \phi(x')\rangle,$$

onde $\phi$ é a expansão quadrática. Isso não é uma analogia — é uma igualdade, e
dá para conferir.

In [ ]:
Xc, yc = make_circles(n_samples=300, factor=0.45, noise=0.10, random_state=1)

# (a) kernel polinomial de grau 2, com gamma=1 e coef0=1
kernel = SVC(kernel="poly", degree=2, gamma=1.0, coef0=1.0, C=5).fit(Xc, yc)

# (b) a expansao explicita das mesmas coordenadas, com um kernel LINEAR
#     (x1, x2) -> (1, x1, x2, x1^2, x1 x2, x2^2), com os pesos binomiais certos
def expandir(M):
    x1, x2 = M[:, 0], M[:, 1]
    return np.c_[np.ones(len(M)), np.sqrt(2) * x1, np.sqrt(2) * x2,
                 x1 ** 2, np.sqrt(2) * x1 * x2, x2 ** 2]


explicito = SVC(kernel="linear", C=5).fit(expandir(Xc), yc)

d_kernel = kernel.decision_function(Xc)
d_explic = explicito.decision_function(expandir(Xc))
print(f"predicoes iguais? {np.array_equal(kernel.predict(Xc), explicito.predict(expandir(Xc)))}")
print(f"maior diferenca na funcao de decisao: {np.abs(d_kernel - d_explic).max():.2e}")
print(f"correlacao entre as duas: {np.corrcoef(d_kernel, d_explic)[0,1]:.10f}")

As duas rotas dão a mesma coisa. Em duas dimensões a expansão tem 6 colunas e o
truque parece desnecessário; em 100 dimensões com grau 3 seriam mais de 170 mil
colunas, e com o kernel RBF seriam **infinitas** — é aí que o truque deixa de ser
elegância e vira a única possibilidade.

In [ ]:
fig, axes = subplots(1, 4, figsize=(11, 3.0))
Xl, yl = make_moons(n_samples=300, noise=0.22, random_state=2)
for ax, (nome, m) in zip(axes, [
        ("linear", SVC(kernel="linear", C=1)),
        ("polinomial grau 3", SVC(kernel="poly", degree=3, C=1, gamma="scale")),
        ("RBF (gamma=1)", SVC(kernel="rbf", gamma=1, C=1)),
        ("RBF (gamma=50)", SVC(kernel="rbf", gamma=50, C=1))]):
    m.fit(Xl, yl)
    desenhar_fronteira(ax, m, Xl, yl, f"{nome}\ntreino {m.score(Xl, yl):.3f}")

O último painel é a lição de sempre, com roupa nova: $\gamma$ grande faz o kernel
RBF enxergar cada ponto isoladamente, e a fronteira vira um conjunto de ilhas em
torno das observações de treino. Acurácia perfeita no treino, e nada aprendido.
O $\gamma$ do RBF é um botão de flexibilidade, exatamente como o grau do polinômio
da Aula 01 e o $1/k$ do KNN da Aula 04.

---
## 5. Por que a grade tem de ser bidimensional

$C$ e $\gamma$ **interagem**: um $\gamma$ maior pode ser compensado por um $C$
menor, e vice-versa. Buscar um de cada vez encontra um ponto ruim. Vamos varrer os
dois e olhar a superfície.

In [ ]:
Xg, yg = make_moons(n_samples=400, noise=0.3, random_state=5)
Cs = np.logspace(-2, 3, 12)
gammas = np.logspace(-2, 2, 12)

busca = skm.GridSearchCV(Pipeline([("sc", StandardScaler()), ("svc", SVC())]),
                         {"svc__C": Cs, "svc__gamma": gammas},
                         cv=5, scoring="accuracy", n_jobs=-1).fit(Xg, yg)
grade = -np.array(busca.cv_results_["mean_test_score"]).reshape(len(Cs), len(gammas))

fig, ax = subplots(figsize=(5.2, 4.0))
im = ax.pcolormesh(gammas, Cs, -grade, shading="auto", cmap="viridis")
ax.plot(busca.best_params_["svc__gamma"], busca.best_params_["svc__C"], "r*", ms=16)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("gamma"); ax.set_ylabel("C")
ax.set_title("acuracia de CV (a estrela e' o otimo)", fontsize=9)
fig.colorbar(im, ax=ax)

print(f"melhor: C = {busca.best_params_['svc__C']:.4g}, "
      f"gamma = {busca.best_params_['svc__gamma']:.4g}, "
      f"acuracia {busca.best_score_:.4f}")

A região boa é uma **faixa diagonal**, não uma cruz. Se você fixasse $\gamma$ no
valor padrão e buscasse só o $C$, andaria por uma linha horizontal do mapa — e
poderia atravessá-lo inteiro sem tocar na faixa. É por isso que a grade é
bidimensional.

E note o `StandardScaler` no `Pipeline`: o kernel RBF depende de
$\|x - x'\|^2$, que soma todas as coordenadas. Sem padronizar, a coluna de maior
amplitude decide o valor do kernel sozinha, e o $\gamma$ ótimo passa a depender das
unidades em que os dados foram medidos.

> **Sua vez.** Refaça a busca com `scoring="roc_auc"` em vez de `"accuracy"`. O
> par $(C, \gamma)$ escolhido muda? Note que o `SVC` não tem `predict_proba` por
> padrão — descubra o que o `scikit-learn` usa como escore nesse caso, e por que
> isso é suficiente para a AUC.

---
## 6. A perda *hinge*, e a probabilidade que não existe

A SVM minimiza a perda ***hinge***, $\max(0, 1 - y f(x))$ com $y \in \{-1, +1\}$.
Compare com a logística, $\log(1 + e^{-y f(x)})$, e com a perda 0–1, que é a que
realmente queremos minimizar e é intratável.

In [ ]:
u = np.linspace(-3, 3, 400)
fig, ax = subplots(figsize=(5.4, 3.2))
ax.plot(u, np.maximum(0, 1 - u), label="hinge (SVM)", lw=1.8)
ax.plot(u, np.log2(1 + np.exp(-u)), label="logistica (em log2)", lw=1.8)
ax.plot(u, (u <= 0).astype(float), label="0-1 (o que queremos)", lw=1.8, ls="--")
ax.axvline(1, ls=":", color="gray")
ax.set_xlabel("margem  y * f(x)"); ax.set_ylabel("perda")
ax.set_ylim(-0.1, 3.2); ax.legend(fontsize=8)

As duas primeiras são aproximações convexas da terceira, e a diferença entre elas é
tudo. A *hinge* **zera** a partir da margem 1: um ponto classificado com folga não
contribui em nada, e é exatamente por isso que a Seção 2 pôde jogar fora 57
observações. A logística nunca zera — ela sempre quer empurrar o ponto para mais
longe, e é essa cauda que sustenta a interpretação probabilística.

Consequência direta: a SVM não estima $P(Y=1\mid x)$. O `scikit-learn` oferece
`probability=True`, que ajusta uma sigmoide sobre a função de decisão por validação
cruzada interna (*Platt scaling*) — mais lento, e um remendo, não uma dedução.
Vamos ver quanto custa e quanto vale.

In [ ]:
import time

Xp, yp = make_moons(n_samples=1500, noise=0.35, random_state=7)
Xp_te, yp_te = make_moons(n_samples=8000, noise=0.35, random_state=8)

modelos = {
    "SVM (probability=False)": SVC(C=1, gamma="scale"),
    "SVM (probability=True)": SVC(C=1, gamma="scale", probability=True, random_state=0),
    "logistica": Pipeline([("poli", PolynomialFeatures(3)),
                           ("sc", StandardScaler()),
                           ("lg", LogisticRegression(max_iter=5000))]),
}
linhas = []
for nome, m in modelos.items():
    t0 = time.perf_counter(); m.fit(Xp, yp); dt = time.perf_counter() - t0
    escore = (m.predict_proba(Xp_te)[:, 1] if hasattr(m, "predict_proba")
              and getattr(m, "probability", True) else m.decision_function(Xp_te))
    linha = {"modelo": nome, "segundos": dt,
             "acuracia": accuracy_score(yp_te, m.predict(Xp_te)),
             "AUC": roc_auc_score(yp_te, escore), "Brier": np.nan}
    if getattr(m, "probability", True) and hasattr(m, "predict_proba"):
        linha["Brier"] = brier_score_loss(yp_te, m.predict_proba(Xp_te)[:, 1])
    linhas.append(linha)
pd.DataFrame(linhas).set_index("modelo").round(4)

A AUC é praticamente a mesma com e sem `probability=True` — o *ordenamento* já
estava lá, na função de decisão. O que muda é o custo e a existência de um número
entre 0 e 1 que se possa multiplicar por dinheiro.

Se você só precisa ranquear, use `decision_function` e economize o tempo. Se
precisa de probabilidade calibrada, a Aula 09 já disse o que fazer: meça o Brier
antes de confiar.

---
## 7. Por que a SVM não escala

O treino do `SVC` resolve um problema quadrático com uma variável por observação, e
a matriz de kernel tem $n^2$ entradas. A teoria diz que o custo fica entre $O(n^2)$
e $O(n^3)$. Vamos medir o expoente.

In [ ]:
tamanhos = np.array([2000, 4000, 8000, 16_000, 32_000])
tempos = {"SVC (kernel RBF)": [], "LinearSVC": [], "logistica": []}
for n in tamanhos:
    Xs, ys = make_moons(n_samples=int(n), noise=0.35, random_state=1)
    Xs = StandardScaler().fit_transform(Xs)
    for nome, m in [("SVC (kernel RBF)", SVC(C=1, gamma="scale")),
                    ("LinearSVC", LinearSVC(C=1, max_iter=20000)),
                    ("logistica", LogisticRegression(max_iter=5000))]:
        t0 = time.perf_counter(); m.fit(Xs, ys); tempos[nome].append(time.perf_counter() - t0)

for nome, ts in tempos.items():
    expoente = np.polyfit(np.log(tamanhos), np.log(np.maximum(ts, 1e-6)), 1)[0]
    print(f"{nome:18s} expoente medido: {expoente:.2f}   "
          f"tempo em n=32000: {ts[-1]*1000:8.1f} ms")

In [ ]:
fig, ax = subplots(figsize=(5.2, 3.2))
for nome, ts in tempos.items():
    ax.plot(tamanhos, ts, "o-", ms=4, label=nome)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("n"); ax.set_ylabel("segundos para ajustar")
ax.legend(fontsize=8)

O expoente medido para o `SVC` fica em torno de **2**, como a teoria prevê,
enquanto o `LinearSVC` e a logística crescem **linearmente**. E a diferença absoluta
é brutal: em $n = 32\,000$ o `SVC` leva mais de mil vezes o tempo do `LinearSVC`.
Extrapolando com expoente 2, um ajuste que leva alguns segundos em $n=32\,000$
levaria **horas** em $n = 10^6$ — e isso para um único ajuste, sem contar a busca de
$(C,\gamma)$, que multiplica tudo pelo tamanho da grade vezes o número de dobras.

As saídas, quando o problema fica grande: `LinearSVC` (que resolve o caso linear
por outro algoritmo), aproximações do kernel como o `Nystroem` — que constrói
explicitamente umas centenas de coordenadas aproximando o RBF, e aí um modelo
linear resolve — ou simplesmente trocar de família.

---
## 8. Caso real: de volta ao câncer de mama

A mesma base da Aula 08, para poder comparar diretamente.

In [ ]:
from sklearn.datasets import load_breast_cancer

dados = load_breast_cancer()
X_tr, X_ts, y_tr, y_ts = skm.train_test_split(dados.data, dados.target,
                                              test_size=0.3, random_state=0,
                                              stratify=dados.target)

svm_busca = skm.GridSearchCV(
    Pipeline([("sc", StandardScaler()), ("svc", SVC())]),
    {"svc__C": np.logspace(-2, 3, 10), "svc__gamma": np.logspace(-4, 1, 10),
     "svc__kernel": ["rbf", "linear"]},
    cv=5, scoring="roc_auc", n_jobs=-1).fit(X_tr, y_tr)

print("melhores parametros:", svm_busca.best_params_)
print(f"AUC de CV no vencedor: {svm_busca.best_score_:.4f}")

In [ ]:
comparacao = {
    "SVM (C e gamma por CV)": svm_busca,
    "logistica (C por CV)": skm.GridSearchCV(
        Pipeline([("sc", StandardScaler()),
                  ("lg", LogisticRegression(max_iter=5000))]),
        {"lg__C": np.logspace(-3, 3, 13)}, cv=5, scoring="roc_auc").fit(X_tr, y_tr),
}
linhas = []
for nome, m in comparacao.items():
    escore = m.decision_function(X_ts)          # serve para a AUC nos dois casos
    linhas.append({"modelo": nome,
                   "acuracia": accuracy_score(y_ts, m.predict(X_ts)),
                   "AUC": roc_auc_score(y_ts, escore)})

sv = svm_busca.best_estimator_.named_steps["svc"]
print(f"kernel escolhido: {svm_busca.best_params_['svc__kernel']}")
print(f"vetores de suporte: {int(sv.n_support_.sum())} de {len(y_tr)} observacoes "
      f"({sv.n_support_.sum()/len(y_tr):.1%})\n")
pd.DataFrame(linhas).set_index("modelo").round(4)

A SVM e a logística chegam essencialmente ao mesmo lugar — o que é comum em
problemas com sinal forte e fronteira quase linear. A diferença que sobra é
prática: a logística entrega probabilidades calibradas de graça e ajusta em
milissegundos; a SVM entrega uma fronteira definida por uma fração das observações
e um hiperparâmetro a mais para buscar.

A escolha, então, não sai do desempenho: sai de o que você vai fazer com a saída.

> **Sua vez.** Compare o número de vetores de suporte do modelo vencedor com o que
> se obtém usando $C = 0{,}01$ e $C = 1000$. Depois responda: se você tivesse de
> guardar o modelo em um dispositivo com memória apertada, qual $C$ preferiria, e o
> que estaria trocando?

---
## Resumo

| Conceito | Onde apareceu | O que vimos |
|---|---|---|
| vetores de suporte | §2 | jogar fora 57 de 60 observações mexe 140× menos que tirar 1 vetor de suporte |
| o $C$ | §3 | $C$ menor $\Rightarrow$ margem mais larga e mais vetores de suporte |
| truque do kernel | §4 | kernel polinomial $=$ expansão explícita, até $10^{-14}$ |
| $\gamma$ do RBF | §4 | é um botão de flexibilidade: $\gamma$ grande vira ilhas em torno dos dados |
| grade 2D | §5 | a região boa é uma faixa diagonal — buscar um parâmetro de cada vez erra |
| perda *hinge* | §6 | zera a partir da margem 1, e é daí que vem a esparsidade da §2 |
| probabilidades | §6 | `probability=True` é *Platt scaling*, um remendo; a AUC já estava na `decision_function` |
| custo | §7 | expoente medido $\approx 2$; o `LinearSVC` é linear e mil vezes mais rápido em $n=32\,000$ |
| caso real | §8 | empata com a logística — a escolha sai do uso, não do desempenho |

**Leitura recomendada.** [AME] §8.2 (SVM) e §4.6 (RKHS, que é a formalização do
truque do kernel). [ISLP] Capítulo 9 inteiro: §9.1 (o classificador de margem
máxima), §9.2 (margem flexível e o papel do $C$ — note que a convenção de sinal do
$C$ deles é a **oposta** à do `scikit-learn`), §9.3 (kernels) e §9.5 (a relação com
a regressão logística, que é a nossa Seção 6).

**Para praticar.** `recursos/listas/Lista de exercícios 08.pdf`.

**A seguir.** A Aula 11 fecha o bloco de classificação trazendo o KNN e as árvores
para cá, e põe todas as fronteiras deste bloco lado a lado nos mesmos dados — que é
a melhor maneira de fixar a intuição de "flexível × rígido" que percorreu o curso.